In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"C:\AI-Anomaly-Detection\outputs\dataset_feature_engineering.csv"
)

df["waktu_tercatat"] = pd.to_datetime(
    df["waktu_tercatat"]
)

df = df.sort_values(
    ["session_id", "waktu_tercatat"]
).reset_index(drop=True)

print("Jumlah baris:", len(df))
print("Jumlah kolom:", len(df.columns))

df.head()

Jumlah baris: 38531
Jumlah kolom: 26


,id_monitoring_udara,suhu,kelembaban,pm10,pm25,barometer,tvoc,waktu_tercatat,gap_sec,session_id,...,pm10_duration,pm10_change_flag,pm10_delta,pm25_duration,pm25_change_flag,pm25_delta,gap_panjang_flag,awal_sesi_flag,pm10_delta_next,pm25_delta_next
0,164460,29.39,46,8.0,3.0,701.29,740.63,2026-01-19 11:40:18,NaN,1,...,0.0,1,0.0,0.0,1,0.0,0,1,-3.0,-1.0
1,164461,29.29,46,5.0,2.0,701.27,740.63,2026-01-19 11:41:22,64.0,1,...,0.0,1,-3.0,0.0,1,-1.0,0,0,-4.0,-2.0
2,164462,29.39,45,1.0,0.0,701.26,740.63,2026-01-19 11:42:27,65.0,1,...,0.0,1,-4.0,0.0,1,-2.0,0,0,3.0,1.0
3,164463,29.39,45,4.0,1.0,701.26,740.63,2026-01-19 11:43:31,64.0,1,...,0.0,1,3.0,0.0,1,1.0,0,0,-4.0,-1.0
4,164464,29.29,44,0.0,0.0,701.27,740.63,2026-01-19 11:44:36,65.0,1,...,0.0,1,-4.0,0.0,1,-1.0,0,0,8.0,4.0


In [2]:
# Menyalin dataset agar data awal tetap aman
hasil_deteksi = df.copy()

# Ambang batas awal spike
THRESHOLD_PM10 = 24
THRESHOLD_PM25 = 12

# Kandidat spike berdasarkan perubahan nilai
hasil_deteksi["pm10_spike"] = (
    hasil_deteksi["pm10_delta"].abs() > THRESHOLD_PM10
)

hasil_deteksi["pm25_spike"] = (
    hasil_deteksi["pm25_delta"].abs() > THRESHOLD_PM25
)

# Kandidat spike dari salah satu atau kedua sensor
hasil_deteksi["kandidat_spike"] = (
    hasil_deteksi["pm10_spike"] |
    hasil_deteksi["pm25_spike"]
)

# Menampilkan jumlah kandidat
print("Hasil awal Spike Detection")
print("Kandidat spike PM10:", hasil_deteksi["pm10_spike"].sum())
print("Kandidat spike PM2.5:", hasil_deteksi["pm25_spike"].sum())
print("Total kandidat spike:", hasil_deteksi["kandidat_spike"].sum())

Hasil awal Spike Detection
Kandidat spike PM10: 184
Kandidat spike PM2.5: 188
Total kandidat spike: 216


In [3]:
# Perubahan pada pembacaan berikutnya dalam sesi yang sama
hasil_deteksi["pm10_delta_next"] = (
    hasil_deteksi.groupby("session_id")["pm10_delta"].shift(-1)
)

hasil_deteksi["pm25_delta_next"] = (
    hasil_deteksi.groupby("session_id")["pm25_delta"].shift(-1)
)

# Deteksi pola spike yang langsung berbalik
hasil_deteksi["pm10_spike_reversal"] = (
    (hasil_deteksi["pm10_delta"] > THRESHOLD_PM10) &
    (hasil_deteksi["pm10_delta_next"] < -THRESHOLD_PM10)
)

hasil_deteksi["pm25_spike_reversal"] = (
    (hasil_deteksi["pm25_delta"] > THRESHOLD_PM25) &
    (hasil_deteksi["pm25_delta_next"] < -THRESHOLD_PM25)
)

hasil_deteksi["spike_reversal"] = (
    hasil_deteksi["pm10_spike_reversal"] |
    hasil_deteksi["pm25_spike_reversal"]
)

print(
    "Kandidat spike yang langsung berbalik:",
    hasil_deteksi["spike_reversal"].sum()
)

print(
    "PM10 dan PM2.5 sama-sama berbalik:",
    (
        hasil_deteksi["pm10_spike_reversal"] &
        hasil_deteksi["pm25_spike_reversal"]
    ).sum()
)

Kandidat spike yang langsung berbalik: 59
PM10 dan PM2.5 sama-sama berbalik: 41


In [4]:
# Klasifikasi kandidat spike
hasil_deteksi["kategori_spike"] = "Normal"

hasil_deteksi.loc[
    hasil_deteksi["kandidat_spike"],
    "kategori_spike"
] = "Spike"

hasil_deteksi.loc[
    hasil_deteksi["spike_reversal"],
    "kategori_spike"
] = "Spike Reversal"

# Menampilkan jumlah setiap kategori
print(
    hasil_deteksi["kategori_spike"].value_counts()
)

kategori_spike
Normal            38315
Spike               157
Spike Reversal       59
Name: count, dtype: int64


In [5]:
kolom_tampil = [
    "session_id",
    "waktu_tercatat",
    "pm10",
    "pm10_delta",
    "pm25",
    "pm25_delta",
    "kategori_spike"
]

contoh_spike = hasil_deteksi[
    hasil_deteksi["kategori_spike"] != "Normal"
][kolom_tampil]

contoh_spike.head(15)

,session_id,waktu_tercatat,pm10,pm10_delta,pm25,pm25_delta,kategori_spike
56,1,2026-01-19 13:35:46,57.0,35.0,26.0,15.0,Spike
58,1,2026-01-19 13:37:56,76.0,23.0,37.0,13.0,Spike Reversal
59,1,2026-01-19 13:39:01,17.0,-59.0,10.0,-27.0,Spike
68,1,2026-01-19 13:48:45,8.0,-40.0,3.0,-19.0,Spike
71,1,2026-01-19 13:52:00,50.0,38.0,23.0,19.0,Spike
78,1,2026-01-19 13:59:34,88.0,52.0,46.0,29.0,Spike Reversal
79,1,2026-01-19 14:00:39,29.0,-59.0,16.0,-30.0,Spike
83,1,2026-01-19 14:05:07,16.0,-63.0,9.0,-30.0,Spike
86,1,2026-01-19 14:08:36,87.0,71.0,44.0,36.0,Spike Reversal
87,1,2026-01-19 14:09:41,43.0,-44.0,21.0,-23.0,Spike


In [6]:
import os

os.makedirs(
    r"C:\AI-Anomaly-Detection\outputs",
    exist_ok=True
)

hasil_deteksi.to_csv(
    r"C:\AI-Anomaly-Detection\outputs\hasil_spike_detection.csv",
    index=False
)

print("Hasil Spike Detection berhasil disimpan.")
print("Total data:", len(hasil_deteksi))

Hasil Spike Detection berhasil disimpan.
Total data: 38531


In [11]:
sensor_duration = [
    "tvoc",
    "barometer",
    "pm10",
    "pm25"
]

THRESHOLD_STUCK = 10

for sensor in sensor_duration:
    kolom_duration = f"{sensor}_duration"

    hasil_stuck[f"{sensor}_stuck"] = (
        hasil_stuck[kolom_duration] >= THRESHOLD_STUCK
    )

    print(
        f"Kandidat stuck {sensor}:",
        hasil_stuck[f"{sensor}_stuck"].sum()
    )

Kandidat stuck tvoc: 32539
Kandidat stuck barometer: 0
Kandidat stuck pm10: 35
Kandidat stuck pm25: 112


In [8]:
print("Apakah df tersedia:", "df" in globals())
print("Apakah hasil_stuck tersedia:", "hasil_stuck" in globals())

print("\nKolom duration yang tersedia:")
print([
    kolom for kolom in df.columns
    if kolom.endswith("_duration")
])

Apakah df tersedia: True
Apakah hasil_stuck tersedia: False

Kolom duration yang tersedia:
['tvoc_duration', 'barometer_duration', 'pm10_duration', 'pm25_duration']


In [10]:
# Membuat salinan dataset untuk analisis Stuck
hasil_stuck = df.copy()

print("Jumlah data:", len(hasil_stuck))
print("Variabel hasil_stuck berhasil dibuat.")

Jumlah data: 38531
Variabel hasil_stuck berhasil dibuat.


In [ ]:
sensor_duration = [
    "tvoc",
    "barometer",
    "pm10",
    "pm25"
]

for sensor in sensor_duration:
    kolom = f"{sensor}_duration"

    print(f"\nStatistik durasi {sensor.upper()}")
    print(
        hasil_stuck[kolom].describe(
            percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
        )
    )


Statistik durasi TVOC
count    38531.000000
mean        66.223593
std         64.057397
min          0.000000
50%         47.283333
75%         93.025000
90%        155.083333
95%        196.300000
99%        285.696667
max        429.750000
Name: tvoc_duration, dtype: float64

Statistik durasi BAROMETER
count    38531.000000
mean         0.251610
std          0.623799
min          0.000000
50%          0.000000
75%          0.000000
90%          1.066667
95%          1.100000
99%          3.150000
max          7.866667
Name: barometer_duration, dtype: float64

Statistik durasi PM10
count    38531.000000
mean         0.243024
std          0.864785
min          0.000000
50%          0.000000
75%          0.000000
90%          1.050000
95%          1.083333
99%          3.166667
max         30.666667
Name: pm10_duration, dtype: float64

Statistik durasi PM25
count    38531.000000
mean         0.745369
std          1.584431
min          0.000000
50%          0.000000
75%          1.05000

In [ ]:
sensor_duration = [
    "tvoc",
    "barometer",
    "pm10",
    "pm25"
]

for sensor in sensor_duration:
    kolom = f"{sensor}_duration"

    batas_95 = hasil_stuck[kolom].quantile(0.95)
    batas_99 = hasil_stuck[kolom].quantile(0.99)

    jumlah_95 = (hasil_stuck[kolom] > batas_95).sum()
    jumlah_99 = (hasil_stuck[kolom] > batas_99).sum()

    print(f"\nSensor: {sensor.upper()}")
    print(f"Batas persentil 95: {batas_95:.2f} menit")
    print(f"Kandidat di atas P95: {jumlah_95}")
    print(f"Batas persentil 99: {batas_99:.2f} menit")
    print(f"Kandidat di atas P99: {jumlah_99}")


Sensor: TVOC
Batas persentil 95: 196.30 menit
Kandidat di atas P95: 1926
Batas persentil 99: 285.70 menit
Kandidat di atas P99: 386

Sensor: BAROMETER
Batas persentil 95: 1.10 menit
Kandidat di atas P95: 1853
Batas persentil 99: 3.15 menit
Kandidat di atas P99: 340

Sensor: PM10
Batas persentil 95: 1.08 menit
Kandidat di atas P95: 1856
Batas persentil 99: 3.17 menit
Kandidat di atas P99: 356

Sensor: PM25
Batas persentil 95: 3.38 menit
Kandidat di atas P95: 1865
Batas persentil 99: 6.61 menit
Kandidat di atas P99: 386


In [ ]:
sensor_duration = [
    "tvoc",
    "barometer",
    "pm10",
    "pm25"
]

for sensor in sensor_duration:
    kolom = f"{sensor}_duration"
    batas_p99 = hasil_stuck[kolom].quantile(0.99)

    kandidat = hasil_stuck[
        hasil_stuck[kolom] > batas_p99
    ].copy()

    kandidat = kandidat.sort_values(
        kolom, ascending=False
    )

    print(f"\n10 Kandidat Stuck Terlama: {sensor.upper()}")

    print(
        kandidat[
            [
                "session_id",
                "waktu_tercatat",
                sensor,
                kolom
            ]
        ].head(10).to_string(index=False)
    )


10 Kandidat Stuck Terlama: TVOC
 session_id      waktu_tercatat  tvoc  tvoc_duration
         47 2026-11-03 01:16:37 731.7     429.750000
         47 2026-11-03 01:15:32 731.7     428.666667
         47 2026-11-03 01:14:27 731.7     427.583333
         47 2026-11-03 01:13:22 731.7     426.500000
         47 2026-11-03 01:12:18 731.7     425.433333
         47 2026-11-03 01:11:13 731.7     424.350000
         47 2026-11-03 01:10:08 731.7     423.266667
         47 2026-11-03 01:09:03 731.7     422.183333
         47 2026-11-03 01:07:59 731.7     421.116667
         47 2026-11-03 01:06:54 731.7     420.033333

10 Kandidat Stuck Terlama: BAROMETER
 session_id      waktu_tercatat  barometer  barometer_duration
         40 2026-08-29 08:06:10     705.67            7.866667
          5 2026-03-27 01:42:45     701.95            7.033333
          7 2026-04-22 08:57:43     702.73            7.033333
         43 2026-09-09 09:24:57     701.53            6.916667
         40 2026-08-29 08:05:04

In [ ]:
sensor_duration = [
    "tvoc",
    "barometer",
    "pm10",
    "pm25"
]

ringkasan_stuck = []

for sensor in sensor_duration:

    gap = (
        hasil_stuck.groupby("session_id")["waktu_tercatat"]
        .diff()
        .dt.total_seconds()
    )

    awal_pengulangan = (
        hasil_stuck["session_id"].ne(
            hasil_stuck["session_id"].shift()
        )
        | hasil_stuck[sensor].ne(
            hasil_stuck[sensor].shift()
        )
        | (gap > 300)
    )

    kelompok = awal_pengulangan.cumsum()

    data_sensor = hasil_stuck.copy()
    data_sensor["kelompok"] = kelompok

    ringkasan = (
        data_sensor.groupby(
            ["session_id", "kelompok"]
        )
        .agg(
            nilai=(sensor, "first"),
            waktu_awal=("waktu_tercatat", "min"),
            waktu_akhir=("waktu_tercatat", "max"),
            jumlah_pembacaan=(sensor, "count"),
            durasi_maksimum=(f"{sensor}_duration", "max")
        )
        .reset_index()
    )

    batas = hasil_stuck[
        f"{sensor}_duration"
    ].quantile(0.99)

    kandidat = ringkasan[
        ringkasan["durasi_maksimum"] > batas
    ].copy()

    kandidat["sensor"] = sensor

    ringkasan_stuck.append(kandidat)

hasil_kelompok_stuck = pd.concat(
    ringkasan_stuck,
    ignore_index=True
)

print("Jumlah kejadian kandidat stuck per sensor:")

print(
    hasil_kelompok_stuck.groupby("sensor").size()
)

hasil_kelompok_stuck.head(15)

Jumlah kejadian kandidat stuck per sensor:
sensor
barometer    260
pm10         227
pm25         165
tvoc          11
dtype: int64


,session_id,kelompok,nilai,waktu_awal,waktu_akhir,jumlah_pembacaan,durasi_maksimum,sensor
0,7,196,722.78,2026-04-21 20:09:57,2026-04-22 00:58:56,202,288.983333,tvoc
1,10,295,713.87,2026-05-05 19:45:20,2026-05-06 00:50:57,253,305.616667,tvoc
2,11,339,731.70,2026-05-08 01:24:58,2026-05-08 06:25:10,247,300.200000,tvoc
3,15,425,159.13,2026-05-19 01:10:35,2026-05-19 06:04:01,237,293.433333,tvoc
4,18,515,740.63,2026-06-04 12:51:47,2026-06-04 17:40:42,257,288.916667,tvoc
5,19,573,848.34,2026-06-08 23:31:25,2026-06-09 05:42:45,327,371.333333,tvoc
6,25,657,722.78,2026-06-19 00:14:54,2026-06-19 05:57:58,303,343.066667,tvoc
7,27,713,1347.66,2026-06-27 00:28:21,2026-06-27 06:11:34,310,343.216667,tvoc
8,46,944,704.97,2026-10-28 19:41:26,2026-10-29 00:36:34,280,295.133333,tvoc
9,46,945,713.87,2026-10-29 00:37:37,2026-10-29 05:45:51,294,308.233333,tvoc


In [ ]:
# Ringkasan jumlah kejadian kandidat stuck
ringkasan_jumlah = (
    hasil_kelompok_stuck
    .groupby("sensor")
    .agg(
        jumlah_kejadian=("kelompok", "count"),
        durasi_terpanjang=("durasi_maksimum", "max"),
        rata_rata_durasi=("durasi_maksimum", "mean"),
        pembacaan_terbanyak=("jumlah_pembacaan", "max")
    )
    .reset_index()
)

ringkasan_jumlah

,sensor,jumlah_kejadian,durasi_terpanjang,rata_rata_durasi,pembacaan_terbanyak
0,barometer,260,7.866667,3.818269,7
1,pm10,227,30.666667,4.412335,27
2,pm25,165,32.233333,9.067172,28
3,tvoc,11,429.750000,324.353030,399


In [ ]:
# Menampilkan seluruh kandidat TVOC
hasil_kelompok_stuck[
    hasil_kelompok_stuck["sensor"] == "tvoc"
].sort_values(
    "durasi_maksimum",
    ascending=False
)

,session_id,kelompok,nilai,waktu_awal,waktu_akhir,jumlah_pembacaan,durasi_maksimum,sensor
10,47,977,731.70,2026-11-02 18:06:52,2026-11-03 01:16:37,399,429.750000,tvoc
5,19,573,848.34,2026-06-08 23:31:25,2026-06-09 05:42:45,327,371.333333,tvoc
7,27,713,1347.66,2026-06-27 00:28:21,2026-06-27 06:11:34,310,343.216667,tvoc
6,25,657,722.78,2026-06-19 00:14:54,2026-06-19 05:57:58,303,343.066667,tvoc
9,46,945,713.87,2026-10-29 00:37:37,2026-10-29 05:45:51,294,308.233333,tvoc
1,10,295,713.87,2026-05-05 19:45:20,2026-05-06 00:50:57,253,305.616667,tvoc
2,11,339,731.70,2026-05-08 01:24:58,2026-05-08 06:25:10,247,300.200000,tvoc
8,46,944,704.97,2026-10-28 19:41:26,2026-10-29 00:36:34,280,295.133333,tvoc
3,15,425,159.13,2026-05-19 01:10:35,2026-05-19 06:04:01,237,293.433333,tvoc
0,7,196,722.78,2026-04-21 20:09:57,2026-04-22 00:58:56,202,288.983333,tvoc


In [ ]:
for sensor in ["pm10", "pm25"]:

    print(f"\n10 Kejadian Stuck Terlama: {sensor.upper()}")

    tampil = hasil_kelompok_stuck[
        hasil_kelompok_stuck["sensor"] == sensor
    ].sort_values(
        "durasi_maksimum",
        ascending=False
    )

    print(
        tampil[
            [
                "session_id",
                "nilai",
                "waktu_awal",
                "waktu_akhir",
                "jumlah_pembacaan",
                "durasi_maksimum"
            ]
        ].head(10).to_string(index=False)
    )


10 Kejadian Stuck Terlama: PM10
 session_id  nilai          waktu_awal         waktu_akhir  jumlah_pembacaan  durasi_maksimum
          4    0.0 2026-02-12 20:48:24 2026-02-12 21:19:04                27        30.666667
          4    0.0 2026-02-12 22:58:54 2026-02-12 23:26:50                22        27.933333
          4    0.0 2026-02-12 21:24:27 2026-02-12 21:35:46                 9        11.316667
          4    0.0 2026-02-12 23:33:32 2026-02-12 23:44:46                 9        11.233333
         41   44.0 2026-09-01 23:54:54 2026-09-02 00:03:30                 4         8.600000
          4    0.0 2026-02-13 00:00:34 2026-02-13 00:08:20                 7         7.766667
         14   24.0 2026-05-13 20:28:33 2026-05-13 20:36:10                 7         7.616667
          4    0.0 2026-02-13 00:10:29 2026-02-13 00:17:44                 4         7.250000
         40   19.0 2026-08-28 14:19:26 2026-08-28 14:26:19                 5         6.883333
          9   25.0 2026-04-

In [ ]:
hasil_kelompok_stuck["jenis_nilai"] = np.where(
    hasil_kelompok_stuck["nilai"] == 0,
    "Nilai nol",
    "Nilai nonnol"
)

ringkasan_nilai_stuck = (
    hasil_kelompok_stuck
    .groupby(["sensor", "jenis_nilai"])
    .agg(
        jumlah_kejadian=("kelompok", "count"),
        durasi_terpanjang=("durasi_maksimum", "max"),
        rata_rata_durasi=("durasi_maksimum", "mean")
    )
    .reset_index()
)

ringkasan_nilai_stuck

,sensor,jenis_nilai,jumlah_kejadian,durasi_terpanjang,rata_rata_durasi
0,barometer,Nilai nonnol,260,7.866667,3.818269
1,pm10,Nilai nol,12,30.666667,10.272222
2,pm10,Nilai nonnol,215,8.600000,4.085271
3,pm25,Nilai nol,7,32.233333,20.533333
4,pm25,Nilai nonnol,158,16.016667,8.559177
5,tvoc,Nilai nonnol,11,429.750000,324.353030


In [ ]:
sensor_duration = [
    "tvoc",
    "barometer",
    "pm10",
    "pm25"
]

ambang_durasi = [5, 10, 15, 30, 60]

semua_kelompok = []

for sensor in sensor_duration:

    gap = (
        hasil_stuck.groupby("session_id")["waktu_tercatat"]
        .diff()
        .dt.total_seconds()
    )

    awal = (
        hasil_stuck["session_id"].ne(
            hasil_stuck["session_id"].shift()
        )
        | hasil_stuck[sensor].ne(
            hasil_stuck[sensor].shift()
        )
        | (gap > 300)
    )

    kelompok = awal.cumsum()

    data = hasil_stuck.copy()
    data["kelompok"] = kelompok

    ringkasan = (
        data.groupby(["session_id", "kelompok"])
        .agg(
            nilai=(sensor, "first"),
            durasi=(f"{sensor}_duration", "max")
        )
        .reset_index()
    )

    ringkasan["sensor"] = sensor
    semua_kelompok.append(ringkasan)

seluruh_kejadian = pd.concat(
    semua_kelompok,
    ignore_index=True
)

for ambang in ambang_durasi:
    print(f"\nAmbang durasi > {ambang} menit")

    print(
        seluruh_kejadian[
            seluruh_kejadian["durasi"] > ambang
        ].groupby("sensor").size()
    )


Ambang durasi > 5 menit
sensor
barometer     24
pm10          37
pm25         400
tvoc         564
dtype: int64

Ambang durasi > 10 menit
sensor
pm10      4
pm25     30
tvoc    494
dtype: int64

Ambang durasi > 15 menit
sensor
pm10      2
pm25      6
tvoc    466
dtype: int64

Ambang durasi > 30 menit
sensor
pm10      1
pm25      3
tvoc    409
dtype: int64

Ambang durasi > 60 menit
sensor
tvoc    285
dtype: int64


In [ ]:
ringkasan_ambang = []

for ambang in ambang_durasi:
    jumlah = (
        seluruh_kejadian[
            seluruh_kejadian["durasi"] > ambang
        ]
        .groupby("sensor")
        .size()
    )

    for sensor in sensor_duration:
        ringkasan_ambang.append({
            "sensor": sensor,
            "ambang_menit": ambang,
            "jumlah_kejadian": jumlah.get(sensor, 0)
        })

tabel_ambang = pd.DataFrame(ringkasan_ambang)

tabel_ambang.pivot(
    index="sensor",
    columns="ambang_menit",
    values="jumlah_kejadian"
)

ambang_menit,5,10,15,30,60
sensor,,,,,
barometer,24,0,0,0,0
pm10,37,4,2,1,0
pm25,400,30,6,3,0
tvoc,564,494,466,409,285


In [ ]:
sensor_columns = [
    "suhu",
    "kelembaban",
    "pm10",
    "pm25",
    "barometer",
    "tvoc"
]

hasil_iqr = []

for sensor in sensor_columns:
    Q1 = df[sensor].quantile(0.25)
    Q3 = df[sensor].quantile(0.75)
    IQR = Q3 - Q1

    batas_bawah = Q1 - (1.5 * IQR)
    batas_atas = Q3 + (1.5 * IQR)

    jumlah_bawah = (df[sensor] < batas_bawah).sum()
    jumlah_atas = (df[sensor] > batas_atas).sum()

    hasil_iqr.append({
        "sensor": sensor,
        "batas_bawah": batas_bawah,
        "batas_atas": batas_atas,
        "jumlah_nilai_rendah": jumlah_bawah,
        "jumlah_nilai_tinggi": jumlah_atas,
        "total_kandidat": jumlah_bawah + jumlah_atas
    })

tabel_iqr = pd.DataFrame(hasil_iqr)

tabel_iqr

,sensor,batas_bawah,batas_atas,jumlah_nilai_rendah,jumlah_nilai_tinggi,total_kandidat
0,suhu,14.690,39.330,179,844,1023
1,kelembaban,20.500,96.500,0,0,0
2,pm10,-16.000,80.000,0,148,148
3,pm25,-4.000,36.000,0,300,300
4,barometer,694.660,709.140,2546,4526,7072
5,tvoc,660.335,803.095,4526,3073,7599


In [ ]:
sensor_columns = [
    "suhu",
    "kelembaban",
    "pm10",
    "pm25",
    "barometer",
    "tvoc"
]

for sensor in sensor_columns:
    print(f"\n===== {sensor.upper()} =====")

    print("5 nilai terendah:")
    print(df[sensor].nsmallest(5).values)

    print("5 nilai tertinggi:")
    print(df[sensor].nlargest(5).values)


===== SUHU =====
5 nilai terendah:
[13.8 13.8 13.8 13.8 13.8]
5 nilai tertinggi:
[44.78 44.78 44.69 44.59 44.59]

===== KELEMBABAN =====
5 nilai terendah:
[21 21 21 21 21]
5 nilai tertinggi:
[79 79 79 79 79]

===== PM10 =====
5 nilai terendah:
[0. 0. 0. 0. 0.]
5 nilai tertinggi:
[489. 353. 224. 220. 186.]

===== PM25 =====
5 nilai terendah:
[0. 0. 0. 0. 0.]
5 nilai tertinggi:
[221. 174. 112. 109.  93.]

===== BAROMETER =====
5 nilai terendah:
[650.43 650.44 650.45 650.46 650.46]
5 nilai tertinggi:
[758.11 758.1  758.09 758.09 758.09]

===== TVOC =====
5 nilai terendah:
[83.45 83.45 83.45 83.45 83.45]
5 nilai tertinggi:
[1366.63 1366.63 1366.63 1366.63 1366.63]


In [ ]:
sensor_columns = [
    "suhu",
    "kelembaban",
    "pm10",
    "pm25",
    "barometer",
    "tvoc"
]

for sensor in sensor_columns:
    print(f"\n===== {sensor.upper()} =====")

    kolom = [
        "session_id",
        "waktu_tercatat",
        sensor
    ]

    print("\n5 nilai terendah:")
    display(
        df.nsmallest(5, sensor)[kolom]
    )

    print("\n5 nilai tertinggi:")
    display(
        df.nlargest(5, sensor)[kolom]
    )


===== SUHU =====

5 nilai terendah:


,session_id,waktu_tercatat,suhu
24409,27,2026-06-27 06:01:01,13.8
24410,27,2026-06-27 06:02:05,13.8
24411,27,2026-06-27 06:03:08,13.8
24412,27,2026-06-27 06:04:11,13.8
24413,27,2026-06-27 06:05:15,13.8



5 nilai tertinggi:


,session_id,waktu_tercatat,suhu
19547,21,2026-06-10 11:20:24,44.78
19548,21,2026-06-10 11:21:28,44.78
19546,21,2026-06-10 11:19:21,44.69
19545,21,2026-06-10 11:18:18,44.59
19549,21,2026-06-10 11:22:31,44.59



===== KELEMBABAN =====

5 nilai terendah:


,session_id,waktu_tercatat,kelembaban
25178,40,2026-08-28 12:53:43,21
25181,40,2026-08-28 12:58:24,21
25182,40,2026-08-28 12:59:32,21
25183,40,2026-08-28 13:00:41,21
25184,40,2026-08-28 13:01:47,21


... (14 outputs left out)


===== TVOC =====

5 nilai terendah:


,session_id,waktu_tercatat,tvoc
35192,48,2026-11-04 20:45:03,83.45
35193,48,2026-11-04 20:46:07,83.45
35194,48,2026-11-04 20:47:11,83.45
35195,48,2026-11-04 20:48:16,83.45
35196,48,2026-11-04 20:49:20,83.45



5 nilai tertinggi:


,session_id,waktu_tercatat,tvoc
23569,27,2026-06-26 14:14:28,1366.63
23570,27,2026-06-26 14:15:31,1366.63
23571,27,2026-06-26 14:16:35,1366.63
23572,27,2026-06-26 14:17:38,1366.63
23575,27,2026-06-26 14:20:47,1366.63


In [ ]:
df.loc[
    (df["session_id"] == 27) &
    (df["waktu_tercatat"].between(
        "2026-06-26 14:05:00",
        "2026-06-26 14:30:00"
    )),
    [
        "session_id",
        "waktu_tercatat",
        "tvoc",
        "tvoc_delta",
        "tvoc_duration"
    ]
]


,session_id,waktu_tercatat,tvoc,tvoc_delta,tvoc_duration
23562,27,2026-06-26 14:05:48,1357.14,0.00,85.266667
23563,27,2026-06-26 14:06:52,1357.14,0.00,86.333333
23564,27,2026-06-26 14:09:12,1357.14,0.00,88.666667
23565,27,2026-06-26 14:10:16,1357.14,0.00,89.733333
23566,27,2026-06-26 14:11:19,1357.14,0.00,90.783333
23567,27,2026-06-26 14:12:22,1357.14,0.00,91.833333
23568,27,2026-06-26 14:13:25,1357.14,0.00,92.883333
23569,27,2026-06-26 14:14:28,1366.63,9.49,0.000000
23570,27,2026-06-26 14:15:31,1366.63,0.00,1.050000
23571,27,2026-06-26 14:16:35,1366.63,0.00,2.116667


In [ ]:
df["tvoc"].value_counts().head(10)

tvoc
731.70    5131
722.78    4510
713.87    3573
740.63    3531
749.56    3034
704.97    2361
758.50    2104
767.45    1787
696.08    1385
776.40    1162
Name: count, dtype: int64

In [ ]:
batas_atas_tvoc = 803.095

print("Jumlah nilai TVOC 1366.63:")
print((df["tvoc"] == 1366.63).sum())

print("\nJumlah nilai TVOC di atas batas IQR:")
print((df["tvoc"] > batas_atas_tvoc).sum())

print("\n10 nilai TVOC tertinggi yang paling sering muncul:")
print(
    df.loc[df["tvoc"] > batas_atas_tvoc, "tvoc"]
    .value_counts()
    .head(10)
)

Jumlah nilai TVOC 1366.63:
83

Jumlah nilai TVOC di atas batas IQR:
3073

10 nilai TVOC tertinggi yang paling sering muncul:
tvoc
848.34     475
1347.66    460
1338.18    384
803.32     319
857.37     258
839.32     252
1357.14    251
821.30     189
812.31     178
866.40      97
Name: count, dtype: int64


In [ ]:
batas_atas_tvoc = 803.095

tvoc_tinggi = df[
    df["tvoc"] > batas_atas_tvoc
].copy()

ringkasan_tvoc_tinggi = (
    tvoc_tinggi.groupby("session_id")
    .agg(
        jumlah_pembacaan=("tvoc", "count"),
        nilai_maksimum=("tvoc", "max"),
        nilai_rata_rata=("tvoc", "mean"),
        jumlah_nilai_unik=("tvoc", "nunique")
    )
    .sort_values(
        "jumlah_pembacaan",
        ascending=False
    )
    .reset_index()
)

ringkasan_tvoc_tinggi.head(15)

,session_id,jumlah_pembacaan,nilai_maksimum,nilai_rata_rata,jumlah_nilai_unik
0,27,1217,1366.63,1347.310789,5
1,19,1075,866.40,850.022344,4
2,16,764,830.31,812.971257,4
3,6,10,803.32,803.320000,1
4,20,7,848.34,848.340000,1


In [ ]:
total_per_sesi = (
    df.groupby("session_id")
    .size()
    .rename("total_pembacaan")
)

ringkasan_tvoc_sesi = ringkasan_tvoc_tinggi.merge(
    total_per_sesi,
    on="session_id"
)

ringkasan_tvoc_sesi["persentase_tinggi"] = (
    ringkasan_tvoc_sesi["jumlah_pembacaan"]
    / ringkasan_tvoc_sesi["total_pembacaan"]
    * 100
)

ringkasan_tvoc_sesi[
    [
        "session_id",
        "total_pembacaan",
        "jumlah_pembacaan",
        "persentase_tinggi",
        "nilai_maksimum",
        "nilai_rata_rata"
    ]
]

,session_id,total_pembacaan,jumlah_pembacaan,persentase_tinggi,nilai_maksimum,nilai_rata_rata
0,27,1217,1217,100.000000,1366.63,1347.310789
1,19,1075,1075,100.000000,866.40,850.022344
2,16,1095,764,69.771689,830.31,812.971257
3,6,1191,10,0.839631,803.32,803.320000
4,20,7,7,100.000000,848.34,848.340000


In [ ]:
hasil_iqr_sesi = []

for session, data in df.groupby("session_id"):

    Q1 = data["tvoc"].quantile(0.25)
    Q3 = data["tvoc"].quantile(0.75)
    IQR = Q3 - Q1

    batas_bawah = Q1 - (1.5 * IQR)
    batas_atas = Q3 + (1.5 * IQR)

    jumlah_anomali = (
        (data["tvoc"] < batas_bawah) |
        (data["tvoc"] > batas_atas)
    ).sum()

    hasil_iqr_sesi.append({
        "session_id": session,
        "jumlah_data": len(data),
        "Q1": Q1,
        "Q3": Q3,
        "batas_bawah": batas_bawah,
        "batas_atas": batas_atas,
        "jumlah_kandidat": jumlah_anomali
    })

tabel_iqr_sesi = pd.DataFrame(hasil_iqr_sesi)

tabel_iqr_sesi.sort_values(
    "jumlah_kandidat",
    ascending=False
)

,session_id,jumlah_data,Q1,Q3,batas_bawah,batas_atas,jumlah_kandidat
16,17,1302,731.70,740.630,718.3050,754.0250,101
24,25,979,722.78,731.700,709.4000,745.0800,93
4,5,1211,731.70,740.630,718.3050,754.0250,74
46,47,1273,731.70,740.630,718.3050,754.0250,70
28,29,85,696.08,696.080,696.0800,696.0800,13
23,24,60,722.78,722.780,722.7800,722.7800,8
33,34,22,740.63,740.630,740.6300,740.6300,2
29,30,52,696.08,696.080,696.0800,696.0800,1
3,4,746,749.56,758.500,736.1500,771.9100,0
2,3,282,767.45,785.370,740.5700,812.2500,0


In [ ]:
batas_sesi = tabel_iqr_sesi[
    ["session_id", "batas_bawah", "batas_atas"]
]

data_tvoc_iqr = df.merge(
    batas_sesi,
    on="session_id",
    how="left"
)

data_tvoc_iqr["unusual_tvoc"] = (
    (data_tvoc_iqr["tvoc"] < data_tvoc_iqr["batas_bawah"])
    | (data_tvoc_iqr["tvoc"] > data_tvoc_iqr["batas_atas"])
)

kandidat_tvoc = data_tvoc_iqr[
    data_tvoc_iqr["unusual_tvoc"]
]

kandidat_tvoc.groupby("session_id").agg(
    jumlah_kandidat=("tvoc", "count"),
    nilai_minimum=("tvoc", "min"),
    nilai_maksimum=("tvoc", "max"),
    rata_rata=("tvoc", "mean")
).sort_values(
    "jumlah_kandidat",
    ascending=False
)

,jumlah_kandidat,nilai_minimum,nilai_maksimum,rata_rata
session_id,,,,
17,101,758.50,758.50,758.5000
25,93,749.56,749.56,749.5600
5,74,758.50,758.50,758.5000
47,70,758.50,758.50,758.5000
29,13,687.19,687.19,687.1900
24,8,713.87,731.70,718.3275
34,2,731.70,731.70,731.7000
30,1,704.97,704.97,704.9700


In [ ]:
for session in [17, 25, 5, 47]:
    print(f"\n===== SESSION {session} =====")

    print(
        df.loc[
            df["session_id"] == session,
            "tvoc"
        ].value_counts().head(10)
    )


===== SESSION 17 =====
tvoc
731.70    614
722.78    212
749.56    196
740.63    179
758.50    101
Name: count, dtype: int64

===== SESSION 25 =====
tvoc
722.78    508
713.87    203
731.70     99
749.56     93
740.63     76
Name: count, dtype: int64

===== SESSION 5 =====
tvoc
731.70    532
740.63    241
722.78    189
749.56    175
758.50     74
Name: count, dtype: int64

===== SESSION 47 =====
tvoc
731.70    678
740.63    279
722.78    144
749.56    102
758.50     70
Name: count, dtype: int64


In [ ]:
for sensor in ["pm10", "pm25"]:

    print(f"\n===== {sensor.upper()} =====")

    print(
        df[sensor]
        .value_counts()
        .head(10)
    )

    print("\n5 nilai tertinggi:")
    print(
        df[sensor]
        .value_counts()
        .sort_index(ascending=False)
        .head(5)
    )


===== PM10 =====
pm10
40.0    1555
16.0    1505
24.0    1486
19.0    1411
32.0    1408
48.0    1106
18.0    1067
43.0    1008
17.0     864
20.0     844
Name: count, dtype: int64

5 nilai tertinggi:
pm10
489.0    1
353.0    1
224.0    1
220.0    1
186.0    1
Name: count, dtype: int64

===== PM25 =====
pm25
17.0    2722
11.0    2651
16.0    2437
18.0    2177
10.0    2032
19.0    1930
22.0    1928
12.0    1919
9.0     1878
21.0    1757
Name: count, dtype: int64

5 nilai tertinggi:
pm25
221.0    1
174.0    1
112.0    1
109.0    1
93.0     1
Name: count, dtype: int64


In [ ]:
for sensor in ["pm10", "pm25"]:

    print(f"\n===== 10 NILAI TERTINGGI {sensor.upper()} =====")

    kolom = [
        "session_id",
        "waktu_tercatat",
        sensor,
        f"{sensor}_delta"
    ]

    print(
        df.nlargest(10, sensor)[kolom].to_string(index=False)
    )


===== 10 NILAI TERTINGGI PM10 =====
 session_id      waktu_tercatat  pm10  pm10_delta
         23 2026-06-15 13:17:41 489.0       470.0
         23 2026-06-15 11:08:54 353.0       304.0
         40 2026-08-29 01:32:46 224.0       165.0
         14 2026-05-14 05:26:51 220.0       115.0
         14 2026-05-14 05:27:54 186.0       -34.0
         23 2026-06-15 10:33:27 175.0       151.0
          8 2026-04-24 02:24:21 147.0        30.0
          8 2026-04-24 02:08:34 146.0        17.0
         18 2026-06-04 16:01:35 145.0        49.0
         23 2026-06-16 09:15:31 145.0       102.0

===== 10 NILAI TERTINGGI PM25 =====
 session_id      waktu_tercatat  pm25  pm25_delta
         23 2026-06-15 13:17:41 221.0       211.0
         23 2026-06-15 11:08:54 174.0       151.0
         40 2026-08-29 01:32:46 112.0        85.0
         14 2026-05-14 05:26:51 109.0        56.0
         14 2026-05-14 05:27:54  93.0       -16.0
         23 2026-06-15 10:33:27  92.0        81.0
          8 2026-04-24 02:

In [ ]:
data_konteks = df[
    (df["session_id"] == 23) &
    (df["waktu_tercatat"] >= "2026-06-15 13:10:00") &
    (df["waktu_tercatat"] <= "2026-06-15 13:25:00")
]

print(
    data_konteks[
        [
            "waktu_tercatat",
            "pm10",
            "pm10_delta",
            "pm25",
            "pm25_delta"
        ]
    ].to_string(index=False)
)

     waktu_tercatat  pm10  pm10_delta  pm25  pm25_delta
2026-06-15 13:10:19  11.0        -4.0   5.0        -2.0
2026-06-15 13:11:22  18.0         7.0   9.0         4.0
2026-06-15 13:12:25  16.0        -2.0   8.0        -1.0
2026-06-15 13:13:28  17.0         1.0  10.0         2.0
2026-06-15 13:14:31  19.0         2.0  10.0         0.0
2026-06-15 13:15:35  17.0        -2.0   8.0        -2.0
2026-06-15 13:16:38  19.0         2.0  10.0         2.0
2026-06-15 13:17:41 489.0       470.0 221.0       211.0
2026-06-15 13:18:44  35.0      -454.0  17.0      -204.0
2026-06-15 13:21:04  16.0       -19.0   9.0        -8.0
2026-06-15 13:22:07  15.0        -1.0   9.0         0.0
2026-06-15 13:23:11  17.0         2.0  10.0         1.0
2026-06-15 13:24:14  16.0        -1.0   9.0        -1.0


In [ ]:
for sensor in ["suhu", "kelembaban"]:

    print(f"\n===== {sensor.upper()} =====")

    print("10 nilai paling sering muncul:")
    print(df[sensor].value_counts().head(10))

    print("\n5 nilai terendah:")
    print(df[sensor].value_counts().sort_index().head(5))

    print("\n5 nilai tertinggi:")
    print(df[sensor].value_counts().sort_index(ascending=False).head(5))


===== SUHU =====
10 nilai paling sering muncul:
suhu
23.93    746
23.83    635
24.03    631
23.63    561
25.22    524
24.73    521
24.13    498
24.83    497
23.53    494
24.33    491
Name: count, dtype: int64

5 nilai terendah:
suhu
13.8     9
13.9    15
14.0    13
14.1    27
14.2    25
Name: count, dtype: int64

5 nilai tertinggi:
suhu
44.78    2
44.69    1
44.59    3
44.49    5
44.39    9
Name: count, dtype: int64

===== KELEMBABAN =====
10 nilai paling sering muncul:
kelembaban
65    1736
67    1670
71    1666
61    1623
64    1500
62    1497
66    1393
63    1316
68    1296
72    1296
Name: count, dtype: int64

5 nilai terendah:
kelembaban
21      6
22     56
23     53
24    105
25    114
Name: count, dtype: int64

5 nilai tertinggi:
kelembaban
79     59
78    218
77    135
76    289
75    635
Name: count, dtype: int64


In [ ]:
for sensor in ["suhu", "kelembaban"]:

    print(f"\n===== 10 NILAI TERTINGGI {sensor.upper()} =====")

    kolom = [
        "session_id",
        "waktu_tercatat",
        sensor
    ]

    print(
        df.nlargest(10, sensor)[kolom].to_string(index=False)
    )

    print(f"\n===== 10 NILAI TERENDAH {sensor.upper()} =====")

    print(
        df.nsmallest(10, sensor)[kolom].to_string(index=False)
    )


===== 10 NILAI TERTINGGI SUHU =====
 session_id      waktu_tercatat  suhu
         21 2026-06-10 11:20:24 44.78
         21 2026-06-10 11:21:28 44.78
         21 2026-06-10 11:19:21 44.69
         21 2026-06-10 11:18:18 44.59
         21 2026-06-10 11:22:31 44.59
         21 2026-06-10 11:23:34 44.59
         21 2026-06-10 11:14:05 44.49
         21 2026-06-10 11:15:08 44.49
         21 2026-06-10 11:16:11 44.49
         21 2026-06-10 11:24:37 44.49

===== 10 NILAI TERENDAH SUHU =====
 session_id      waktu_tercatat  suhu
         27 2026-06-27 06:01:01  13.8
         27 2026-06-27 06:02:05  13.8
         27 2026-06-27 06:03:08  13.8
         27 2026-06-27 06:04:11  13.8
         27 2026-06-27 06:05:15  13.8
         27 2026-06-27 06:06:18  13.8
         27 2026-06-27 06:07:21  13.8
         27 2026-06-27 06:08:24  13.8
         27 2026-06-27 06:09:28  13.8
         27 2026-06-27 05:49:25  13.9

===== 10 NILAI TERTINGGI KELEMBABAN =====
 session_id      waktu_tercatat  kelembaban
    

In [ ]:
print("===== 10 NILAI PALING SERING MUNCUL =====")
print(df["barometer"].value_counts().head(10))

print("\n===== 5 NILAI TERENDAH =====")
print(df["barometer"].value_counts().sort_index().head(5))

print("\n===== 5 NILAI TERTINGGI =====")
print(
    df["barometer"]
    .value_counts()
    .sort_index(ascending=False)
    .head(5)
)

===== 10 NILAI PALING SERING MUNCUL =====
barometer
701.62    137
702.07    133
701.53    132
701.80    129
701.65    124
702.19    119
701.59    118
702.58    114
701.56    113
703.36    112
Name: count, dtype: int64

===== 5 NILAI TERENDAH =====
barometer
650.43    1
650.44    1
650.45    1
650.46    8
650.47    6
Name: count, dtype: int64

===== 5 NILAI TERTINGGI =====
barometer
758.11    1
758.10    1
758.09    3
758.08    4
758.07    1
Name: count, dtype: int64


In [ ]:
for sensor in ["barometer"]:

    print(f"\n===== 10 NILAI TERTINGGI {sensor.upper()} =====")

    kolom = [
        "session_id",
        "waktu_tercatat",
        sensor
    ]

    print(
        df.nlargest(10, sensor)[kolom].to_string(index=False)
    )

    print(f"\n===== 10 NILAI TERENDAH {sensor.upper()} =====")

    print(
        df.nsmallest(10, sensor)[kolom].to_string(index=False)
    )


===== 10 NILAI TERTINGGI BAROMETER =====
 session_id      waktu_tercatat  barometer
         48 2026-11-04 21:33:52     758.11
         48 2026-11-04 21:32:48     758.10
         48 2026-11-04 21:31:39     758.09
         48 2026-11-04 21:47:53     758.09
         48 2026-11-04 21:48:58     758.09
         48 2026-11-04 21:34:56     758.08
         48 2026-11-04 21:36:00     758.08
         48 2026-11-04 21:46:49     758.08
         48 2026-11-04 21:50:02     758.08
         48 2026-11-04 21:30:35     758.07

===== 10 NILAI TERENDAH BAROMETER =====
 session_id      waktu_tercatat  barometer
         27 2026-06-26 14:31:19     650.43
         27 2026-06-26 14:45:02     650.44
         27 2026-06-26 14:36:37     650.45
         27 2026-06-26 14:21:50     650.46
         27 2026-06-26 14:28:09     650.46
         27 2026-06-26 14:29:13     650.46
         27 2026-06-26 14:32:23     650.46
         27 2026-06-26 14:35:33     650.46
         27 2026-06-26 14:38:44     650.46
         27 20

In [ ]:
sensor_columns = [
    "suhu",
    "kelembaban",
    "pm10",
    "pm25",
    "barometer",
    "tvoc"
]

hasil_iqr = []

for sensor in sensor_columns:

    Q1 = df[sensor].quantile(0.25)
    Q3 = df[sensor].quantile(0.75)
    IQR = Q3 - Q1

    batas_bawah = Q1 - (1.5 * IQR)
    batas_atas = Q3 + (1.5 * IQR)

    hasil_iqr.append({
        "sensor": sensor,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "batas_bawah": batas_bawah,
        "batas_atas": batas_atas,
        "jumlah_kandidat": (
            (df[sensor] < batas_bawah) |
            (df[sensor] > batas_atas)
        ).sum()
    })

tabel_iqr = pd.DataFrame(hasil_iqr)

print(tabel_iqr.round(3).to_string(index=False))

    sensor     Q1     Q3   IQR  batas_bawah  batas_atas  jumlah_kandidat
      suhu  23.93  30.09  6.16       14.690      39.330             1023
kelembaban  49.00  68.00 19.00       20.500      96.500                0
      pm10  20.00  44.00 24.00      -16.000      80.000              148
      pm25  11.00  21.00 10.00       -4.000      36.000              300
 barometer 700.09 703.71  3.62      694.660     709.140             7072
      tvoc 713.87 749.56 35.69      660.335     803.095             7599


In [ ]:
hasil_unusual = df.copy()

for _, row in tabel_iqr.iterrows():

    sensor = row["sensor"]
    batas_bawah = row["batas_bawah"]
    batas_atas = row["batas_atas"]

    hasil_unusual[f"unusual_{sensor}"] = (
        (hasil_unusual[sensor] < batas_bawah) |
        (hasil_unusual[sensor] > batas_atas)
    )

kolom_flag = [
    f"unusual_{sensor}"
    for sensor in sensor_columns
]

hasil_unusual["unusual_value"] = (
    hasil_unusual[kolom_flag].any(axis=1)
)

print("Jumlah kandidat Unusual Value:")
print(hasil_unusual["unusual_value"].sum())

print("\nJumlah kandidat per sensor:")
print(hasil_unusual[kolom_flag].sum())

Jumlah kandidat Unusual Value:
8281

Jumlah kandidat per sensor:
unusual_suhu          1023
unusual_kelembaban       0
unusual_pm10           148
unusual_pm25           300
unusual_barometer     7072
unusual_tvoc          7599
dtype: int64


In [ ]:
kolom_gabungan = kolom_flag + ["unusual_value"]

for kolom in kolom_gabungan:
    hasil_deteksi[kolom] = hasil_unusual[kolom]

print("Jumlah baris:", len(hasil_deteksi))

print("\nKolom deteksi:")
print([
    kolom for kolom in hasil_deteksi.columns
    if kolom.startswith("unusual_")
])

Jumlah baris: 38531

Kolom deteksi:
['unusual_suhu', 'unusual_kelembaban', 'unusual_pm10', 'unusual_pm25', 'unusual_barometer', 'unusual_tvoc', 'unusual_value']


In [ ]:
threshold_stuck = {
    "barometer": 10,
    "pm10": 15,
    "pm25": 15,
    "tvoc": 60
}

for sensor, batas in threshold_stuck.items():

    hasil_deteksi[f"stuck_{sensor}"] = (
        hasil_deteksi[f"{sensor}_duration"] > batas
    )

kolom_stuck = [
    f"stuck_{sensor}"
    for sensor in threshold_stuck
]

hasil_deteksi["stuck_value"] = (
    hasil_deteksi[kolom_stuck].any(axis=1)
)

print("Jumlah kandidat Stuck Value:")
print(hasil_deteksi["stuck_value"].sum())

print("\nJumlah kandidat per sensor:")
print(hasil_deteksi[kolom_stuck].sum())

Jumlah kandidat Stuck Value:
15724

Jumlah kandidat per sensor:
stuck_barometer        0
stuck_pm10            23
stuck_pm25            42
stuck_tvoc         15696
dtype: int64


In [ ]:
hasil_deteksi["rule_based_anomaly"] = (
    hasil_deteksi["kandidat_spike"] |
    hasil_deteksi["stuck_value"] |
    hasil_deteksi["unusual_value"]
)

print("===== HASIL GABUNGAN RULE-BASED =====")

print("Total data:", len(hasil_deteksi))

print(
    "Kandidat Spike:",
    hasil_deteksi["kandidat_spike"].sum()
)

print(
    "Kandidat Stuck Value:",
    hasil_deteksi["stuck_value"].sum()
)

print(
    "Kandidat Unusual Value:",
    hasil_deteksi["unusual_value"].sum()
)

print(
    "Total kandidat Rule-Based:",
    hasil_deteksi["rule_based_anomaly"].sum()
)

print(
    "Data normal:",
    (~hasil_deteksi["rule_based_anomaly"]).sum()
)

===== HASIL GABUNGAN RULE-BASED =====
Total data: 38531
Kandidat Spike: 216
Kandidat Stuck Value: 15724
Kandidat Unusual Value: 8281
Total kandidat Rule-Based: 20879
Data normal: 17652


In [ ]:
metode = [
    "kandidat_spike",
    "stuck_value",
    "unusual_value"
]

print("===== IRISAN HASIL DETEKSI =====")

for kolom in metode:
    print(
        f"{kolom}:",
        hasil_deteksi[kolom].sum()
    )

print("\n===== KOMBINASI DETEKSI =====")

hasil_deteksi["kombinasi_deteksi"] = (
    hasil_deteksi[metode]
    .apply(
        lambda row: " + ".join(
            [
                nama for nama, aktif
                in zip(
                    ["Spike", "Stuck", "Unusual"],
                    row
                )
                if aktif
            ]
        ) or "Normal",
        axis=1
    )
)

print(
    hasil_deteksi["kombinasi_deteksi"]
    .value_counts()
    .to_string()
)

===== IRISAN HASIL DETEKSI =====
kandidat_spike: 216
stuck_value: 15724
unusual_value: 8281

===== KOMBINASI DETEKSI =====
kombinasi_deteksi
Normal                     17652
Stuck                      12480
Unusual                     5020
Stuck + Unusual             3163
Spike                         71
Spike + Unusual               64
Spike + Stuck                 47
Spike + Stuck + Unusual       34


In [ ]:
import os

os.makedirs(
    r"C:\AI-Anomaly-Detection\outputs",
    exist_ok=True
)

path_output = (
    r"C:\AI-Anomaly-Detection\outputs"
    r"\hasil_rule_based_detection.csv"
)

hasil_deteksi.to_csv(
    path_output,
    index=False
)

print("Dataset berhasil disimpan!")
print("Lokasi:", path_output)
print("Jumlah baris:", len(hasil_deteksi))
print("Jumlah kolom:", len(hasil_deteksi.columns))

Dataset berhasil disimpan!
Lokasi: C:\AI-Anomaly-Detection\outputs\hasil_rule_based_detection.csv
Jumlah baris: 38531
Jumlah kolom: 47
